In [ ]:
# Install dependencies
!pip -q install llama-index pymupdf
!pip -q install llama-index-llms-gemini llama-index-embeddings-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [5]:
# Imports
import os
import fitz  # PyMuPDF

from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer

from llama_index.llms.gemini import Gemini
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [8]:
# Load my key from secrets, assert if fail, print if success
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
assert "GOOGLE_API_KEY" in os.environ, "api key failed"
print("GOOGLE_API_KEY loaded into environment variables.")

GOOGLE_API_KEY loaded into environment variables.


In [9]:
# Upload pdf
from google.colab import files

uploaded = files.upload()  # upload the Lender Fee Worksheet PDF
pdf_path = next(iter(uploaded.keys()))
print("Using PDF:", pdf_path)

Saving LenderFeesWorksheetNew-2.pdf to LenderFeesWorksheetNew-2.pdf
Using PDF: LenderFeesWorksheetNew-2.pdf


In [10]:
# Parse PDF
def load_pdf_as_documents(pdf_path: str) -> list[Document]:
    pdf = fitz.open(pdf_path)
    docs = []

    for i, page in enumerate(pdf):
        text = page.get_text("text").strip()
        if text:
            # store per-page so retrieval can cite page-like chunks naturally
            docs.append(Document(text=text, metadata={"page": i + 1}))
    return docs

documents = load_pdf_as_documents(pdf_path)
print("Pages loaded:", len(documents))
print("Example metadata:", documents[0].metadata if documents else None)

Pages loaded: 1
Example metadata: {'page': 1}


In [12]:
# Chunking configuration
# Baseline: sentence-based chunks with overlap (good general default) aka semantic chunking
chunk_size = 512
chunk_overlap = 100

splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
print("Chunking:", {"chunk_size": chunk_size, "chunk_overlap": chunk_overlap})

Chunking: {'chunk_size': 512, 'chunk_overlap': 100}


In [30]:
!pip -q install -U llama-index-llms-google-genai google-genai

import os
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Orignal code has an older model and it didn't work, and was using and old endpoint.
# If for whatever reason this cell fails, check if a new endpoint is being used or the model type, aw man

## Gemini LLM for response generation
llm = GoogleGenAI(
    model="gemini-2.5-flash",
    api_key=os.environ["GOOGLE_API_KEY"],
)

print("Models set (GoogleGenAI).")

# Embeddings (fast and solid quality, good model)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Settings.embed_model = embed_model
Settings.llm = llm



Models set (GoogleGenAI).


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
# Build my nodes and make the vector index

nodes = splitter.get_nodes_from_documents(documents)
print("Total chunks (nodes):", len(nodes))

index = VectorStoreIndex(nodes)
print("Index built.")


Total chunks (nodes): 2
Index built.


In [35]:
# Retrieval method: vector baseline with ranking params

top_k = 5  # retrieval depth; can be tuned

retriever = VectorIndexRetriever(index=index, similarity_top_k=top_k)

print("Retriever ready:", {"method": "vector", "top_k": top_k})

Retriever ready: {'method': 'vector', 'top_k': 5}


In [36]:
# RAG Query Engine

synthesizer = get_response_synthesizer(response_mode="compact")

query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
)

print("RAG pipeline ready.")

RAG pipeline ready.


In [37]:
# Debug helper, shows retrieved chunks

def debug_retrieval(q: str, k: int = None):
    k = k or top_k
    retriever.similarity_top_k = k

    results = retriever.retrieve(q)
    print("QUERY:", q)
    print("Retrieved:", len(results), "chunks\n")

    for i, r in enumerate(results, 1):
        meta = r.node.metadata
        score = getattr(r, "score", None)
        preview = r.node.text[:300].replace("\n", " ")
        print(f"{i}) score={score:.4f} page={meta.get('page')}  preview={preview}...\n")

debug_retrieval("What is the total estimated monthly payment?", k=5)

QUERY: What is the total estimated monthly payment?
Retrieved: 2 chunks

1) score=0.7170 page=1  preview=Your actual rate, payment, and cost could be higher. Get an official Loan Estimate before choosing a loan. Fee Details and Summary Applicants: Application No: Date Prepared: Loan Program: Prepared By: THIS IS NOT A GOOD FAITH ESTIMATE (GFE). This "Fees Worksheet" is provided for informational purpos...

2) score=0.6644 page=1  preview=000.00 1,121.53 4,520.00 380,000.00 Cash Deposit 5,000.00 needed to close 95,641.53 1,869.37 39.58 400.00 2,308.95 ORIGINATION CHARGES Underwriting Fee XYZ Lender Borrower $ 550.00 Wire Transfer Fee XYZ Lender Borrower $ 75.00 Administration Fee XYZ Lender Borrower $ 445.00 OTHER CHARGES Appraisal F...



Example Prompts

In [38]:
!pip -q install nest_asyncio

import nest_asyncio
nest_asyncio.apply()

print("nest_asyncio applied")

nest_asyncio applied


In [41]:
prompt1 = "What is the total estimated monthly payment?"
response1 = await query_engine.aquery(prompt1)

print("User:", prompt1)
print("System:", response1)

User: What is the total estimated monthly payment?
System: The total estimated monthly payment is $2,308.95.


In [42]:
prompt2 = "How much does the borrower pay for lender's title insurance?"
response2 = await query_engine.aquery(prompt2)

print("User:", prompt2)
print("System:", response2)

User: How much does the borrower pay for lender's title insurance?
System: The borrower pays $650.00 for lender's title insurance.
